# Save RR curves as netCDF files

***** EDIT The GBD RR curves can be downloaded [here](https://doi.org/10.6069/vkdr-qy60) in csv format. This script converts them to netCDF.

In [1]:
import os
import re
import glob
import pandas as pd
import config
from utils.utils import require_dir
import pathlib

In [1]:
# === Setup ===
DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "GBD23" / "RR_curves" / "temperature")

# === Health variables ===
health_vars_in = ["ckd", "cvd_cmp", "cvd_htn", "cvd_ihd", "cvd_stroke",
                  "diabetes", "inj_animal", "inj_disaster", "inj_drowning",
                  "inj_homicide", "inj_mech", "inj_othunintent", "inj_suicide",
                  "inj_trans_other", "inj_trans_road", "lri", "resp_copd"]

health_vars_out = ["CHRONIC_KIDNEY_DISEASE", "CARDIOMYOPATHY_MYOCARDITIS",
                   "HYPERTENSIVE_HEART_DISEASE", "ISCHEMIC_HEART_DISEASE",
                   "STROKE", "DIABETES", "INJ_ANIMAL", "INJ_DISASTER",
                   "DROWNING", "HOMICIDE", "MECHANICAL_INJ",
                   "OTHER_UNINTENTIONAL_INJ", "SUICIDE", "OTHER_TRANSPORT_INJ",
                   "ROAD_TRANSPORT_INJ", "LOWER_RESPIRATORY_INFECTIONS", "COPD"]

# === Loop through variables ===
for i in range(len(health_vars_in)):
    var_in = health_vars_in[i]
    var_out = health_vars_out[i]
    pattern = os.path.join(DIR, f"{var_in}_curve_samples.csv")
    matches = glob.glob(pattern)

    if len(matches) == 0:
        raise FileNotFoundError(f"No file found for variable: {var_in}")
    if len(matches) > 1:
        raise ValueError(f"Multiple files matched for variable {var_in}: {matches}")

    df = pd.read_csv(matches[0])

    # fix: round out float noise before using as an index
    df["daily_temperature"] = df["daily_temperature"].round(1)
    # (annual_temperature is already clean int64, no rounding needed)

    # sanity check: rounding should not merge two originally-distinct rows
    dup = df.duplicated(subset=["annual_temperature", "daily_temperature"], keep=False)
    if dup.any():
        raise ValueError(f"{var_in}: rounding created {dup.sum()} colliding rows — check precision")

    draw_cols = [c for c in df.columns if c.startswith("draw_")]

    long_df = df.melt(
        id_vars=["annual_temperature", "daily_temperature"],
        value_vars=draw_cols,
        var_name="draws",
        value_name="value",
    )
    long_df["draws"] = long_df["draws"].str.replace("draw_", "", regex=False).astype(int)

    da = (
        long_df
        .set_index(["annual_temperature", "daily_temperature", "draws"])["value"]
        .to_xarray()
    )

    da.name = "logRR_curve"
    da.attrs["cause_variable"] = var_out
    da.attrs["description"] = (
        "Burkart K, Brauer M, Aravkin A et al. Estimating the "
        "cause-specific relative risks of non-optimal temperature "
        "on daily mortality: a two-part modelling approach applied "
        "to the Global Burden of Disease Study"
    )

    out_path = os.path.join(DIR, f"{var_out}_logRR_curve_samples.nc")
    da.to_netcdf(out_path)
    print(f"Saved {out_path} shape={da.shape}")

print("All processing complete.")

SyntaxError: unterminated string literal (detected at line 2) (693153468.py, line 2)